# Tables

On this page you can find all tables and code to produce the tables of
the manuscript.

## Results

### Table 1

In [3]:
#%% Load packages
import os, sys, pickle
import numpy as np
import pandas as pd
from great_tables import GT
from pathlib import Path

# Paths
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = baseDir / 'data'
funcDir = baseDir / 'analysis' / 'functions'
sys.path.append(str(funcDir))

from table_formatting import format_mean_std, format_str

#%% Initialize dataframe
filepath = os.path.join(dataDir,'isometric-measurements.csv')
df = pd.read_csv(filepath, index_col=0)

mean_row = df.mean()
std_row = df.std()

# Create a formatted row "avg ± std"
avg_std_row_rounded = pd.Series({
    'fmax [N]': format_mean_std(df['fmax [N]'],n=-2, pm="$\pm$"),
    'lce_opt [cm]': format_mean_std(df['lce_opt [cm]'],n=1, pm="$\pm$"),
    'lsee0 [cm]': format_mean_std(df['lsee0 [cm]'],n=0, pm="$\pm$"),
    'rmsd [%]': format_mean_std(df['rmsd [%]'],n=1, pm="$\pm$"),
}, name='AVG±SD')


df_rounded = df.copy()
df_rounded['fmax [N]']    = format_str(df_rounded['fmax [N]'], n=-2)
df_rounded['lce_opt [cm]'] = format_str(df_rounded['lce_opt [cm]'], n=1)
df_rounded['lsee0 [cm]']   = format_str(df_rounded['lsee0 [cm]'], n=0)
df_rounded['rmsd [%]']    = format_str(df_rounded['rmsd [%]'], n=1)

df_final = pd.concat([df_rounded, avg_std_row_rounded.to_frame().T])

#%% TeX table
from gt_tex import make_latex, insert_rows, delete_rows

gt_df = df_final.copy()
gt_df = gt_df.reset_index()
gt_df.columns = ['Participant','$F_{CE}^{max}$ [N]','$L_{CE}^{opt}$ [cm]', '$L_{SEE}^{0}$ [cm]','RMSD [%]']
gt_table = (GT(gt_df)
            .cols_align(align='center') 
            .cols_align(align='left',columns='Participant') 
)

# Transform to LateX table
latex_str = make_latex(gt_table.as_latex())
latex_str = delete_rows(latex_str, row_numbers=[0])
add_rows = {
    0: r" \bfseries Participant & $\bm{F_{CE}^{max}}$ [N] & $\bm{L_{CE}^{opt}}$ [cm] & $\bm{L_{SEE}^{0}}$ [cm] & \bfseries RMSD [\%] \\ \hline",
#     2: r"  & \bfseries 0.40 & \bfseries 0.35C & \bfseries 0.30 & \bfseries 0.25 & \bfseries 0.20 & \bfseries 0.35A & \bfseries 0.35B & \bfseries 0.35D & \bfseries 0.35E \\ \hline",
#     8: r" \end{tabularx}"
}
latex_str = insert_rows(latex_str, add_rows)
latex_str += (r"\break\hfill\footnotesize{"+ 
              r"$F_{CE}^{max}$, $L_{CE}^{opt}$ and $L_{SEE}^{0}$ were estimated from the isometric measurements for each participant individually. "
              r"The root mean square difference (RMSD) was normalised by the individual maximal predicted knee joint moment, which is the product "
              r"of $F_{CE}^{max}$ and the moment arm of \textit{m. quadriceps femoris.}"
              "}")

# Write to a .tex file
with open("tbl-r-muspar.tex", "w", encoding="utf-8") as f:
    f.write(latex_str)

#%% Great table
from great_tables import GT

gt_df = df_final.copy()
gt_df = gt_df.reset_index()
gt_df.columns = ['Participant','$F_{CE}^{max}$ [N]','$L_{CE}^{opt}$ [cm]', '$L_{SEE}^{0}$ [cm]','RMSD [%]']
gt_table = (GT(gt_df)
    .cols_align(align='center') 
    .cols_align(align='left',columns='Participant') 
)
gt_table


### Table 2

In [5]:
import os, sys
import numpy as np
import pandas as pd
from great_tables import GT
from pathlib import Path

# Paths
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = baseDir / 'data'
funcDir = baseDir / 'analysis' / 'functions'
sys.path.append(str(funcDir))

import kinetics
from table_formatting import format_mean_std

#%% Calculate mechanical work durring passive cycles
cond_names = ['0.20', '0.25', '0.30', '0.35C', '0.40', '0.35A', '0.35B', '0.35D', '0.35E']
W = np.full((4, 9), 'NaN', dtype='U21')
for iCond, cond in enumerate(cond_names):
    Ppos_ext = []
    Ppos_flx = []
    Pneg_ext = []
    Pneg_flx = []
    for iPP,pp in enumerate([1,2,3,4,5,6]):
        filepath = os.path.join(dataDir,'dataExp',f'pp{pp:02d}','AMPO measurements',f'pp{pp:02d}_{cond}_t1.csv')
        df = pd.read_csv(filepath)
        data = df.T.to_numpy()
        
        time,phi,_,torqueComp,_,_,emgQuad,emgHams,phase = data[0:9]
        
        iCycles = [5,6,7,8,9]
        if pp == 3 and cond == '0.35E':
            iCycles = [6,7,8,9,10] # for this condition pp3 started one cycle too late!
        Wnet,Wpos,Wneg = kinetics.compute_work(phi,torqueComp,phase,'c',iCycles)
        _,Wpos_ext,Wneg_ext = kinetics.compute_work(phi,torqueComp,phase,'ext',iCycles)
        _,Wpos_flx,Wneg_flx = kinetics.compute_work(phi,torqueComp,phase,'flx',iCycles)
        
        Ppos_ext.append(Wpos_ext/Wnet*100) # [%]
        Ppos_flx.append(Wpos_flx/Wnet*100) # [%]
        Pneg_ext.append(Wneg_ext/Wnet*100) # [%]
        Pneg_flx.append(Wneg_flx/Wnet*100) # [%]
        
    W[0,iCond] = format_mean_std(Ppos_ext,n=1,pm="$\pm$")
    W[1,iCond] = format_mean_std(Ppos_flx,n=1,pm="$\pm$")
    W[2,iCond] = format_mean_std(Pneg_ext,n=1,pm="$\pm$")
    W[3,iCond] = format_mean_std(Pneg_flx,n=1,pm="$\pm$")

df = pd.DataFrame(
    W,
    index=['Ppos_ext [%]', 'Ppos_flx [%]', 'Pneg_ext [%]', 'Pneg_flx [%]'],
    columns=cond_names
)

#%% TeX table
from gt_tex import make_latex, insert_rows, delete_rows, replace_latex_table_cell

gt_df = df.copy()
gt_df.index = ['$W_{ext}^{+}$', '$W_{flex}^{+}$', '$W_{ext}^{-}$', '$W_{flex}^{-}$']
gt_df = gt_df.reset_index()
conditions = ['0.20', '0.25', '0.30', '0.35C', '0.40', '0.35A', '0.35B', '0.35D', '0.35E']

gt_table = (GT(gt_df)
    .cols_align(align='center') 
    .cols_align(align='left', columns="index")
    .cols_label(index='')
    .tab_spanner(label='Condition', columns=conditions)
)
gt_table

# Transform to LateX table
latex_str = make_latex(gt_table.as_latex())
latex_str = delete_rows(latex_str, row_numbers=[0,1])
add_rows = {
    0: r" & \multicolumn{9}{c|}{\textit{Condition}} \\ \hline",
    1: r" & \bfseries 0.20 & \bfseries 0.25 & \bfseries 0.30 & \bfseries 0.35C & \bfseries 0.40 & \bfseries 0.35A & \bfseries 0.35B & \bfseries 0.35D & \bfseries 0.35E \\ \hline",
}
latex_str = insert_rows(latex_str, add_rows)
latex_str = replace_latex_table_cell(latex_str, r'\bm{$W_{ext}^{+}$}', row=2,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bm{$W_{flex}^{+}$}', row=3,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bm{$W_{ext}^{-}$}', row=4,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bm{$W_{flex}^{-}$}', row=5,col=0)
latex_str += (r"\break\hfill\footnotesize{"+ 
              r"Positive (+) and negative (-) mechanical work during knee " + 
              r"extension (ext) and flexion (flex) were calculated by integrating " + 
              r"the instantaneous mechanical power over time, separately for " + 
              r"the periods when the instantaneous power was positive (positive " +
              r"work) and negative (negative work) respectively. }")
latex_str = latex_str.replace(
    r"\begin{tabular}{|l|c|c|c|c|c|c|c|c|c|}",
    r" \begin{tabularx}{\textwidth}{|p{1.0cm}|*{9}{>{\centering\arraybackslash}X|}}")
latex_str = latex_str.replace(
    r"\end{tabular}",
    r"\end{tabularx}")

# Write to a .tex file
with open("tbl-r-mechwork.tex", "w", encoding="utf-8") as f:
    f.write(latex_str)

#%% Great Table
gt_df = df.copy()
gt_df.index = ['$W_{ext}^{+}$', '$W_{flex}^{+}$', '$W_{ext}^{-}$', '$W_{flex}^{-}$']
gt_df = gt_df.reset_index()
conditions = ['0.40', '0.35C', '0.30', '0.25', '0.20', '0.35A', '0.35B', '0.35D', '0.35E']

gt_table = (GT(gt_df)
    .cols_align(align='center') 
    .cols_align(align='left', columns="index")
    .cols_label(index='')
    .tab_spanner(label='Condition', columns=conditions)
)
gt_table

### Table 3

In [13]:
#%% Load packages
import os, sys
import numpy as np
import pandas as pd
from great_tables import GT
from pathlib import Path

# Paths
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = baseDir / 'data'
funcDir = baseDir / 'analysis' / 'functions'
sys.path.append(str(funcDir))

import kinetics

def fmt_row(row):
    return [f'{x:.2f}' for x in row]

#%% Load condition parameters
# Load conditions
data = pd.read_csv(dataDir / 'conditions.csv', sep=',', header=None, dtype=str).to_numpy()

# Set conditions order
cond_names = ['0.20', '0.25', '0.30', '0.35C', '0.40', '0.35A', '0.35B', '0.35D', '0.35E']

# Previously I had a different order, so we need to re-order first
current_order = data[0, :]  # e.g., ['0.25', '0.20', '0.35C', ...]
# Find indices that would reorder to match condNames
reorder_idx = [np.where(current_order == c)[0][0] for c in cond_names]
# Reorder all columns
data = data[:, reorder_idx]

cf = data[1].astype(float)
fts = data[2].astype(float)
Text = fts / cf
Tflx = (1 - fts) / cf
AMPO_pred = data[3].astype(float)

#%% Experimental AMPO values
# Participant info
day, trial = 4, 1
filepath = os.path.join(dataDir,'isometric-measurements.csv')
df = pd.read_csv(filepath, index_col=0)

pps = [1,2,3,4,5,6] # participant no.
n_pp = len(pps) # amount of participants
fmax = df['fmax [N]'].to_numpy() # [N] est maximal isometric CE force per particiapnt

ampo_exp = np.full((Npp, 9), np.nan)  # participants x conditions

for iPP, pp in enumerate(pps):
    for iCond, cond in enumerate(cond_names):
        dataFldr = os.path.join(dataDir, 'dataExp', f'pp{pp:02d}', 'AMPO measurements', '')
        filename = f'pp{pp:02d}_{cond}_t{trial}'
        df = pd.read_csv(dataFldr + filename + '.csv')
        data = df.T.to_numpy()
        time, phi, _, torqueComp, _, _, emgQuad, emgHams, phase = data[0:9]

        iCycles = [5, 6, 7, 8, 9]
        if pp == 3 and cond == '0.35E':
            iCycles = [6, 7, 8, 9, 10]

        Wnet, Wpos, Wneg = kinetics.compute_work(phi, torqueComp, phase, 'c', iCycles)
        ampo = np.max(Wnet * cf[iCond]) / (fmax[iPP] * 0.093)
        ampo_exp[iPP, iCond] = ampo

#%% Format all rows to 0.2f and build DataFrame
cf_fmt = fmt_row(cf)
fts_fmt = fmt_row(fts)
Text_fmt = fmt_row(Text)
Tflx_fmt = fmt_row(Tflx)
AMPO_pred_fmt = fmt_row(AMPO_pred)
ampo_exp_fmt = [fmt_row(row) for row in ampo_exp]

# Compute mean ± std across participants for each condition
avg_std = [f'{np.nanmean(ampo_exp[:, i]):.2f} ± {np.nanstd(ampo_exp[:, i]):.2f}' for i in range(ampo_exp.shape[1])]

# Stack all rows
all_rows = [cf_fmt, fts_fmt, Text_fmt, Tflx_fmt, *ampo_exp_fmt, avg_std, AMPO_pred_fmt]
row_labels = ['cf', 'fts', 'Text', 'Tflx', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'Average ± SD', 'Predicted']
df = pd.DataFrame(all_rows, index=row_labels, columns=cond_names)

#%% TeX Table
from gt_tex import make_latex, insert_rows, replace_latex_table_cell, delete_rows

gt_df = df.copy()
gt_df.index = ['CF [Hz]', 'FTS [ ]', '$T_{ext}$ [s]', '$T_{flex}$ [s]', 
    'P1 [$s^{-1}$]', 'P2 [$s^{-1}$]', 'P3 [$s^{-1}$]', 'P4 [$s^{-1}$]', 'P5 [$s^{-1}$]', 'P6 [$s^{-1}$]', 'Average ± SD [$s^{-1}$]', 'Predicted [$s^{-1}$]']
gt_df = gt_df.reset_index()

gt_table = (GT(gt_df)
    .cols_align(align='center') 
    .cols_align(align='left', columns="index")
    .cols_label(index='')
    .tab_spanner(label='Condition', columns=cond_names)
)
gt_table

# Transform to LateX table
latex_str = make_latex(gt_table.as_latex())
latex_str = delete_rows(latex_str, row_numbers=[0,1])
add_rows = {
     0: r"  & \multicolumn{9}{|c|}{\itshape Condition} \\ \hline",
     1: r"  & \bfseries 0.20 & \bfseries 0.25 & \bfseries 0.30 & \bfseries 0.35C & \bfseries 0.40 & \bfseries 0.35A & \bfseries 0.35B & \bfseries 0.35D & \bfseries 0.35E \\ \hline",
     2: r" \multicolumn{10}{|r|}{\itshape Movement characteristics} \\ \hline",
     7: r" \multicolumn{10}{|r|}{\itshape AMPO} \\ \hline",
    # 12: r" \end{tabularx}"
}
latex_str = insert_rows(latex_str, add_rows)

latex_str = replace_latex_table_cell(latex_str, r'\bfseries CF [Hz]', row=2,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries FTS [ ]', row=3,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bm{$T_{ext}$} ' + r'\bfseries [s]', row=4,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bm{$T_{flex}$} ' + r'\bfseries [s]', row=5,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Participant 1 [s\textsuperscript{-1}]', row=6,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Participant 2 [s\textsuperscript{-1}]', row=7,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Participant 3 [s\textsuperscript{-1}]', row=8,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Participant 4 [s\textsuperscript{-1}]', row=9,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Participant 5 [s\textsuperscript{-1}]', row=10,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Participant 6 [s\textsuperscript{-1}]', row=11,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Average $\pm$ SD [s\textsuperscript{-1}]', row=12,col=0)
latex_str = replace_latex_table_cell(latex_str, r'\bfseries Predicted [s\textsuperscript{-1}]', row=13,col=0)

latex_str += (r"\break\hfill\footnotesize{"+ 
              r"CF denotes cycle frequency, FTS denotes fraction of cycle time spent shortening, "
              r"$T_{ext}$ and $T_{flex}$ denote the knee joint extension and flexion time per cycle. "
              r"Measured AMPO was normalised to the product of the participant's individually estimated $F_{CE}^{max}$ "
              r"and the generic model parameter value of $L_{CE}^{opt}$, while predicted AMPO was normalised to "
              r"the product of model parameters $F_{CE}^{max}$ and $L_{CE}^{opt}$ (see Methods). }")
latex_str = latex_str.replace(
    r"\begin{tabular}{|l|c|c|c|c|c|c|c|c|c|}",
    r" \begin{tabularx}{\textwidth}{|p{2.8cm}|*{9}{>{\centering\arraybackslash}X|}}")

latex_str = latex_str.replace(
    r"\end{tabular}",
    r"\end{tabularx}")

# Write to a .tex file
with open("tbl-r-overview.tex", "w", encoding="utf-8") as f:
    f.write(latex_str)

#%% Great table
gt_df = df.copy()
gt_df.index = ['CF [Hz]', 'FTS [ ]', '$T_{ext}$ [s]', '$T_{flex}$ [s]', 
    'P1 [$s^{-1}$]', 'P2 [$s^{-1}$]', 'P3 [$s^{-1}$]', 'P4 [$s^{-1}$]', 'P5 [$s^{-1}$]', 'P6 [$s^{-1}$]', 'Average ± SD [$s^{-1}$]', 'Predicted [$s^{-1}$]']
gt_df = gt_df.reset_index()

group = 4*['Movement characteristics'] + 8*['AMPO']
gt_df.insert(0, "Group", group) 

gt_table = (GT(gt_df)
    .tab_stub(rowname_col="index", groupname_col="Group")
    .cols_align(align='center') 
    .cols_align(align='left', columns="index")
    .cols_label(index='')
    .tab_spanner(label='Condition', columns=cond_names)
)
gt_table

## Appendix

### Table A1

In [61]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
from great_tables import GT, loc, style
from pathlib import Path

# Paths
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = baseDir / 'data'
funcDir = baseDir / 'analysis' / 'functions'
sys.path.append(str(funcDir))

#%% Extract parameters
params = np.array([
    ['A0',        '$A_0$',            'm',        'Musculoskeletal geometry',  '@van_soest_which_2000',               r'MTC length at $\phi_{knee}=0$'],
    ['A1',        '$A_1$',            'm/rad',    'Musculoskeletal geometry',  '@van_soest_which_2000',               r'Linear coefficient of MTC-$\phi_{knee}$ relation'],
    ['arel',      '$a^{rel}$',        '-',        'Contraction dynamics',      '@van_soest_which_2000',               r'Hill constant'],
    ['brel',      '$b^{rel}$',        '1/s',      'Contraction dynamics',      '@van_soest_which_2000',               r'Hill constant'],
    ['fmax',      '$F_{CE}^{max}$',   'N',        'Contraction dynamics',      '@van_soest_which_2000',               r'Maximum isometric CE force'],
    ['kpee',      '$k_{PEE}$',        'N/m<sup>2</sup>',   'Contraction dynamics',       '@van_soest_contribution_1993',            r'PEE stiffness scaling parameter'],
    ['ksee',      '$k_{SEE}$',        'N/m<sup>2</sup>',   'Contraction dynamics',      '@van_soest_which_2000',               r'SEE stiffness scaling parameter'],
    ['lce_opt',   '$L_{CE}^{opt}$',   'm',        'Contraction dynamics',      '@van_soest_which_2000',               r'CE optimum length'],
    ['lpee0',     '$L_{PEE}^0$',      'm',        'Contraction dynamics',       '@van_soest_contribution_1993',            r'PEE slack length'],
    ['lsee0',     '$L_{SEE}^0$',      'm',        'Contraction dynamics',      '@van_soest_which_2000',               r'SEE slack length'],
    ['w',         '$w$',              '-',        'Contraction dynamics',      '@van_soest_which_2000',               r'Determines CE force-length relation width'],
    ['a_act',     '$a_{act}$',        '-',        'Excitation dynamics',       '@hatze_myocybernetic_1981',           r'Determines steepness $\gamma$-$q$ relation'],
    ['b_act1',    '$b_{act,1}$',      '-',        'Excitation dynamics',       '@hatze_myocybernetic_1981',           r'Determines $pCa^{2+}$ level at which $q=0.5$'],
    ['b_act2',    '$b_{act,2}$',      '-',        'Excitation dynamics',       '@hatze_myocybernetic_1981',           r'Determines $pCa^{2+}$ level at which $q=0.5$'],
    ['b_act3',    '$b_{act,3}$',      '-',        'Excitation dynamics',       '@hatze_myocybernetic_1981',           r'Determines $pCa^{2+}$ level at which $q=0.5$'],
    ['kCa',       '$kCa$',            'mol/L',    'Excitation dynamics',       '@kistemaker_length-dependent_2005',   r'Relates $\gamma$ to $pCa^{2+}$'],
    ['tact',      '$\\tau_{act}$',    's',        'Excitation dynamics',       '@hatze_myocybernetic_1981',           r'Activation time constant'],
    ['tdeact',    '$\\tau_{deact}$',  's',        'Excitation dynamics',       '@hatze_myocybernetic_1981',           r'De-activation time constant'],
])
variables,symbols,units,partypes,references,descriptions = list(zip(*params))  # This transposes the list of lists

musparms = []
muspar = pickle.load(open(os.path.join(dataDir,'VAS_muspar.pkl'), 'rb'))
muspar['b_act1'], muspar['b_act2'], muspar['b_act3'] = muspar['b_act']
musparms = [muspar[k] for k in params[:,0]]
# musparms = ['0.21', '0.042', '0', '0.41', '5.2', '5250', r'$1.28 \cdot 10^8$', '0.093', '0.16', '0.56', '-4.59', '5.17', '1.08', '-0.19', r'$8.00 \cdot 10^{-6}$', r'$88.9 \cdot 10^{-3}$', r'$88.9 \cdot 10^{-3}$']

#%% Create pandas dataframe
df = pd.DataFrame(musparms, columns=['Value'], index=params[:,0])

df.insert(0, "Description", descriptions) 
df.insert(1, "Symbol", symbols) 
df.insert(2, "Unit", units) 
df.insert(4, "Reference", references) 
df.insert(5, "Partype", partypes) 

#%% Some are in other units so..
for parm in ['A0', 'A1', 'lce_opt', 'lpee0', 'lsee0']: # from m to cm
    df.loc[parm, 'Value'] = df.loc[parm, 'Value'] * 1e2
    df.loc[parm, 'Unit'] = 'cm'
for parm in ['tact', 'tdeact']: # from s to ms
    df.loc[parm, 'Value'] = df.loc[parm, 'Value'] * 1e3
    df.loc[parm, 'Unit'] = 'ms'
df.loc['A1', 'Unit'] = 'cm/rad'

#%% TeX table
from gt_tex import make_latex, insert_rows, fix_reference, delete_rows

df_tex = df.copy()
df_tex = df_tex.drop('Partype', axis=1)

gt_table = (GT(df_tex)
    .cols_align(align='left', columns=["Description","Parameter"])
    .fmt_number(columns=["Value"], n_sigfig=3)
    .fmt_number(columns=["Value"], n_sigfig=2, rows=[2,10,14]) #arel, w, bact3
    .fmt_number(columns=["Value"], n_sigfig=4, rows=[4]) # fmax
    .fmt_scientific(columns=["Value"], n_sigfig=3, rows=[5,6,15]) #kpee,ksee,kca
)
gt_table

citation_map = {
    "van_soest_which_2000": "van Soest \& Casius (2000)",
    "kistemaker_length-dependent_2005": "Kistemaker et al. (2005)",
    "hatze_myocybernetic_1981": "Hatze (1981)",
    "van_soest_contribution_1993": "van Soest \& Bobbert (1993)"
}

# Transform to LateX table
latex_str = make_latex(gt_table.as_latex())
latex_str = delete_rows(latex_str, row_numbers=[0])
add_rows = {
    0: r"  \bfseries Description & \bfseries Symbol & \bfseries Unit & \bfseries Value & \bfseries Reference \\ \hline",
    1: r"  \multicolumn{5}{|r|}{\itshape Musculoskeletal geometry} \\ \hline",
    5: r"  \multicolumn{5}{|r|}{\itshape Contraction dynamics} \\ \hline",
    13: r"  \multicolumn{5}{|r|}{\itshape Excitation dynamics} \\ \hline",
}

latex_str = insert_rows(latex_str, add_rows)
latex_str = fix_reference(latex_str,citation_map)
latex_str += (r"\break\hfill\footnotesize{"+ 
              r"Parameter values are depicted only for those key to predict the maximally attainable AMPO. }")

# Write to a .tex file
with open("apptbl-parms.tex", "w", encoding="utf-8") as f:
    f.write(latex_str)

#%% Great table
citation_map = {
    "van_soest_which_2000": "van Soest & Casius",
    "kistemaker_length-dependent_2005": "Kistemaker et al.",
    "hatze_myocybernetic_1981": "Hatze",
    "van_soest_contribution_1993": "van Soest & Bobbert"
}
def replace_citation(x):
    if x.startswith('@'):
        key = x[1:]
        display = citation_map.get(key, key)

        # extract the year = last underscore-separated part
        parts = key.split('_')
        year = parts[-1] if parts[-1].isdigit() else "year"

        return (
            f'<span class="citation" data-cites="{key}">'
            f'{display} (<a href="#ref-{key}" role="doc-biblioref" aria-expanded="false">{year}</a>)'
            f'</span>'
        )
    return x

df_gt = df.copy()
df_gt['Reference'] = df_gt['Reference'].apply(replace_citation)

gt_table = (GT(df_gt)
    .tab_stub(rowname_col="Description", groupname_col="Partype")
    .cols_align(align='left', columns=["Parameter"])
    .fmt_number(columns=["Value"], n_sigfig=3)
    .fmt_number(columns=["Value"], n_sigfig=2, rows=[2,10,14]) #arel, w, bact3
    .fmt_number(columns=["Value"], n_sigfig=4, rows=[4]) # fmax
    .fmt_scientific(columns=["Value"], n_sigfig=3, rows=[5,6,15]) #kpee,ksee,kca
    .tab_style(style = style.text(style = "italic"), locations = loc.row_groups())
)
gt_table